In [2]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/distance_reranker.py
import math
import sys
from pathlib import Path
from typing import List, Dict, Any, Tuple

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.sidecar_manager import SidecarManager
from src.config import (
    HARD_FILTERING_THRESHOLD,
    RERANKING_TOP_N,
)

class DistanceReranker:
    def __init__(self, sidecar_manager: SidecarManager, hard_filter_threshold = HARD_FILTERING_THRESHOLD):
        """
        :param sidecar_manager: Istanza di SidecarManager per accedere al file JSON.
        :param hard_filter_threshold: Soglia di distance_factor oltre la quale un chunk viene considerato escluso.
        """
        self.sidecar = sidecar_manager
        self.hard_filter_threshold = hard_filter_threshold

    def _extract_chunk_id(self, item: Any) -> str:
        """
        Estrae l'ID univoco del chunk (funziona con oggetti LangChain, dict o tuple).
        L'ID, se non nativamente presente nei metadati, è restituito nel formato: DOCID_chunk_CHUNKINDEX
        """
        doc = item[0] if isinstance(item, (tuple, list)) else item

        if isinstance(doc, dict):
            if "chunk_id" in doc:
                return str(doc["chunk_id"])
            metadata = doc.get("metadata", doc)
            doc_id = metadata.get("doc_id", metadata.get("source", "doc"))
            chunk_idx = metadata.get("chunk_index", 0)
            return f"{doc_id}_chunk_{chunk_idx}"

        if hasattr(doc, "metadata"):
            doc_id = doc.metadata.get("doc_id", doc.metadata.get("source", "doc"))
            chunk_idx = doc.metadata.get("chunk_index", 0)
            return f"{doc_id}_chunk_{chunk_idx}"

        return str(doc)

    def rerank(self, candidates: List[Any], top_n: int = RERANKING_TOP_N) -> List[Dict[str, Any]]:
        """
        Ricalcola i punteggi basandosi sui distance_factor salvati nel sidecar.
        
        :param candidates: Lista di candidate chunk con punteggio iniziale [(doc, score), ...]
        :param top_n: Numero massimo di chunk da selezionare dopo il re-ranking per Ollama.
        :return: Lista dei primi Top-N chunk sopravvissuti, ordinati per score finale.
        """
        sidecar_data = self.sidecar.load_data()
        pairwise_deltas = sidecar_data.get("pairwise_deltas", {})

        parsed_candidates = []
        for item in candidates:
            if isinstance(item, (tuple, list)):
                doc, initial_score = item[0], float(item[1])
            elif isinstance(item, dict):
                doc = item
                initial_score = float(item.get("score", 0.5))
            else:
                doc = item
                initial_score = 0.5

            chunk_id = self._extract_chunk_id(doc)
            parsed_candidates.append({
                "raw_doc": doc,
                "chunk_id": chunk_id,
                "initial_score": initial_score,
                "final_score": initial_score,
                "applied_penalties": [],
                "excluded": False
            })

        # Ricalcolo Score tramite Pairwise Deltas
        for c1 in parsed_candidates:
            id1 = c1["chunk_id"]
            max_penalty = 1.0

            for c2 in parsed_candidates:
                id2 = c2["chunk_id"]
                if id1 == id2:
                    continue

                # Chiave simmetrica A_AND_B
                pair_key = "_AND_".join(sorted([str(id1), str(id2)]))

                if pair_key in pairwise_deltas:
                    delta_info = pairwise_deltas[pair_key]
                    factor = float(delta_info.get("distance_factor", 1.0))

                    if factor > max_penalty:
                        max_penalty = factor

                    c1["applied_penalties"].append({
                        "paired_with": id2,
                        "distance_factor": factor
                    })

            # Hard Filtering o Soft Reweighting
            if max_penalty >= self.hard_filter_threshold:
                c1["excluded"] = True
            else:
                c1["final_score"] = c1["initial_score"] / max_penalty

        # Filtraggio ed estrazione dei Top-N
        valid_candidates = [c for c in parsed_candidates if not c["excluded"]]
        valid_candidates.sort(key=lambda x: x["final_score"], reverse=True)

        return valid_candidates[:top_n]
EOF

In [3]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/frontend/chunk_graph.js
import * as d3 from "https://esm.sh/d3@7";

export function render({ model, el }) {
  el.innerHTML = "";

  const container = d3.select(el)
    .append("div")
    .style("position", "relative")
    .style("width", "650px")
    .style("font-family", "sans-serif");

  const width = 650;
  const height = 420;

  let selectedNodeId = null;

  const svg = container.append("svg")
    .attr("width", width)
    .attr("height", height)
    .style("background", "#f8fafc")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "8px")
    .style("cursor", "grab");

  // Contenitore SVG scalabile e traslabile
  const g = svg.append("g");

  // Gestore Zoom e Pan
  const zoom = d3.zoom()
    .scaleExtent([0.2, 4])
    .on("zoom", (event) => {
      g.attr("transform", event.transform);
    });

  svg.call(zoom);

  const infoBox = container.append("div")
    .style("position", "absolute")
    .style("top", "12px")
    .style("right", "12px")
    .style("width", "230px")
    .style("padding", "10px")
    .style("background", "rgba(255, 255, 255, 0.95)")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "6px")
    .style("font-size", "12px")
    .style("color", "#334155")
    .style("pointer-events", "none")
    .style("box-shadow", "0 2px 4px rgba(0,0,0,0.05)")
    .html("<b>📌 Info Chunk</b><br><span style='color:#94a3b8;'>Clicca un nodo per vederne testo e penalità</span>");

  function draw() {
    const graph = model.get("graph_data");
    if (!graph || !graph.nodes || graph.nodes.length === 0) return;

    // Lettura dinamica della soglia inviata da Python (fallback a 2.5)
    const hardFilterThreshold = graph.hard_filter_threshold || 2.5;

    g.selectAll("*").remove();

    const nodes = graph.nodes.map(d => ({ ...d }));
    const links = graph.links ? graph.links.map(d => ({ ...d })) : [];

    // Trova la penalità massima tra tutti gli archi incidenti sul nodo
    function getMaxPenalty(nodeId) {
      let maxPenalty = 1.0;
      links.forEach(l => {
        const srcId = typeof l.source === 'object' ? l.source.id : l.source;
        const tgtId = typeof l.target === 'object' ? l.target.id : l.target;
        if (srcId === nodeId || tgtId === nodeId) {
          const factor = l.distance_factor || (l.target_distance && l.base_distance ? l.target_distance / l.base_distance : 1.0);
          if (factor > maxPenalty) maxPenalty = factor;
        }
      });
      return maxPenalty;
    }

    // Simulazione D3 basata su distanze da Qdrant * distance_factor
    const simulation = d3.forceSimulation(nodes)
      .force("link", d3.forceLink(links)
        .id(d => d.id)
        .distance(d => (d.base_distance || 120) * (d.distance_factor || 1.0))
      )
      .force("charge", d3.forceManyBody().strength(-180))
      .force("center", d3.forceCenter(width / 2, height / 2));

    const link = g.append("g")
      .selectAll("line")
      .data(links)
      .enter().append("line")
      .attr("stroke", "#94a3b8")
      .attr("stroke-width", 2);

    const linkText = g.append("g")
      .selectAll("text")
      .data(links)
      .enter().append("text")
      .attr("font-size", "11px")
      .attr("font-weight", "bold")
      .attr("fill", "#0284c7")
      .attr("text-anchor", "middle");

    const node = g.append("g")
      .selectAll("circle")
      .data(nodes)
      .enter().append("circle")
      .attr("r", 12)
      .attr("stroke-width", 2)
      .style("cursor", "pointer");

    const label = g.append("g")
      .selectAll("text")
      .data(nodes)
      .enter().append("text")
      .attr("font-size", "11px")
      .attr("dx", 15)
      .attr("dy", 4)
      .attr("fill", "#1e293b");

    function updateNodeStyles() {
      node
        .attr("fill", d => {
          if (d.id === selectedNodeId) return "#f59e0b"; // Giallo/Ambra = Selezionato
          const maxP = getMaxPenalty(d.id);
          if (maxP >= hardFilterThreshold) return "#ef4444"; // Rosso = Escluso
          if (maxP > 1.0) return "#f97316"; // Arancione = Penalizzato
          return "#6366f1"; // Indaco = Normale
        })
        .attr("stroke", d => (d.id === selectedNodeId ? "#1e293b" : "#ffffff"))
        .attr("stroke-width", d => (d.id === selectedNodeId ? 3 : 2));

      label.text(d => {
        const maxP = getMaxPenalty(d.id);
        return maxP > 1.0 ? `${d.id} (${maxP.toFixed(1)}x)` : d.id;
      });
    }

    simulation.on("end", () => {
      nodes.forEach(n => {
        n.fx = n.x;
        n.fy = n.y;
      });
    });

    const drag = d3.drag()
      .on("start", (event, d) => {
        nodes.forEach(n => {
          n.fx = n.x;
          n.fy = n.y;
        });
      })
      .on("drag", (event, d) => {
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;
        updatePositions();
      })
      .on("end", (event, d) => {
        d.fx = event.x;
        d.fy = event.y;
        d.x = event.x;
        d.y = event.y;

        links.forEach(l => {
          const srcId = typeof l.source === 'object' ? l.source.id : l.source;
          const tgtId = typeof l.target === 'object' ? l.target.id : l.target;

          if (srcId === d.id || tgtId === d.id) {
            const dx = l.target.x - l.source.x;
            const dy = l.target.y - l.source.y;
            const currentDist = Math.sqrt(dx * dx + dy * dy);
            const baseDist = l.base_distance || 120;
            const factor = currentDist / baseDist;

            l.distance_factor = factor;

            model.set("pairwise_edit", {
              chunk_1: srcId,
              chunk_2: tgtId,
              distance_factor: factor
            });
            model.save_changes();
          }
        });

        updateNodeStyles();
        updatePositions();
      });

    node.call(drag);

    node.on("click", (event, d) => {
      selectedNodeId = d.id;
      updateNodeStyles();

      const maxP = getMaxPenalty(d.id);
      let statusHtml = "<span style='color:#10b981; font-weight:bold;'>✅ Attivo</span>";
      if (maxP >= hardFilterThreshold) {
        statusHtml = `<span style='color:#ef4444; font-weight:bold;'>❌ ESCLUSO (${maxP.toFixed(2)}x)</span>`;
      } else if (maxP > 1.0) {
        statusHtml = `<span style='color:#f97316; font-weight:bold;'>⚠️ Penalizzato (${maxP.toFixed(2)}x)</span>`;
      }

      infoBox.html(`
        <b>🆔 ${d.id}</b><br>
        <b>Stato:</b> ${statusHtml}<br>
        <hr style="border:0; border-top:1px solid #e2e8f0; margin:6px 0;">
        <span style="color:#334155; display:block; max-height:100px; overflow-y:auto;">📖 ${d.text || "Nessun testo disponibile"}</span>
      `);

      model.set("selected_tag", { id: d.id, text: d.text || "" });
      model.save_changes();
    });

    function updatePositions() {
      link
        .attr("x1", d => d.source.x)
        .attr("y1", d => d.source.y)
        .attr("x2", d => d.target.x)
        .attr("y2", d => d.target.y);

      linkText
        .attr("x", d => (d.source.x + d.target.x) / 2)
        .attr("y", d => (d.source.y + d.target.y) / 2 - 5)
        .text(d => {
          const dx = d.target.x - d.source.x;
          const dy = d.target.y - d.source.y;
          const dist = Math.round(Math.sqrt(dx * dx + dy * dy));
          const baseDist = d.base_distance || 120;
          const factor = (dist / baseDist).toFixed(2);
          return `${dist}px (${factor}x)`;
        });

      node
        .attr("cx", d => d.x)
        .attr("cy", d => d.y);

      label
        .attr("x", d => d.x)
        .attr("y", d => d.y);
    }

    updateNodeStyles();
    simulation.on("tick", updatePositions);
  }

  model.on("change:graph_data", draw);
  draw();
}

export default { render };
EOF

In [4]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/graph_builder.py
import networkx as nx
from src.config import HARD_FILTERING_THRESHOLD, BASE_DISTANCE_PX

class GraphBuilder:
    def __init__(self):
        self.graph = nx.Graph()

    def add_chunk_node(self, chunk_id: str, text: str, metadata: dict = None):
        """Aggiunge un nodo rappresentante un chunk di testo."""
        attrs = {"text": text, "type": "chunk"}
        if metadata:
            attrs.update(metadata)
        self.graph.add_node(chunk_id, **attrs)

    def add_relation(self, source_id: str, target_id: str, similarity: float):
        """
        Crea un arco basato sulla Cosine Similarity calcolata da Qdrant.
        Calcola la distanza base fisica in pixel (D_base = BASE_DISTANCE_PX * (1 - similarity)).
        """
        if source_id in self.graph and target_id in self.graph:
            # Minima distanza consentita per evitare sovrapposizioni (clamp a 0.1)
            distance_scale = max(0.1, 1.0 - float(similarity))
            base_dist = BASE_DISTANCE_PX * distance_scale

            self.graph.add_edge(
                source_id,
                target_id,
                similarity=float(similarity),
                base_distance=round(base_dist, 2)
            )

    def to_json_data(self) -> dict:
        """Esporta il grafo in formato compatibile con D3.js ed inietta la soglia di config."""
        data = nx.node_link_data(self.graph)
        if "edges" in data and "links" not in data:
            data["links"] = data.pop("edges")

        data["hard_filter_threshold"] = HARD_FILTERING_THRESHOLD
        return data
EOF

In [8]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/chunk_widget.py
import sys
from pathlib import Path
import anywidget
import traitlets

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.sidecar_manager import SidecarManager
from src.config import HARD_FILTERING_THRESHOLD

class ChunkGraphWidget(anywidget.AnyWidget):
    _esm = Path(__file__).parent / "frontend" / "chunk_graph.js"

    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_tag = traitlets.Dict({}).tag(sync=True)
    pairwise_edit = traitlets.Dict({}).tag(sync=True)

    def __init__(self, sidecar_path=None, **kwargs):
        super().__init__(**kwargs)
        self.sidecar = SidecarManager(filepath=sidecar_path) if sidecar_path else SidecarManager()
        self.observe(self._on_pairwise_edit, names=["pairwise_edit"])

    def load_graph(self, raw_graph_data: dict):
        """
        Carica la struttura del grafo applicando i distance_factor registrati nel sidecar
        e iniettando la soglia hard_filter_threshold.
        """
        deltas = self.sidecar.data.get("pairwise_deltas", {})
        links = raw_graph_data.get("links", raw_graph_data.get("edges", []))
        enriched_links = []

        for link in links:
            l_copy = dict(link)
            src = l_copy["source"]["id"] if isinstance(l_copy["source"], dict) else l_copy["source"]
            tgt = l_copy["target"]["id"] if isinstance(l_copy["target"], dict) else l_copy["target"]

            k1 = f"{src}_AND_{tgt}"
            k2 = f"{tgt}_AND_{src}"

            factor = 1.0
            if k1 in deltas:
                factor = float(deltas[k1].get("distance_factor", 1.0))
            elif k2 in deltas:
                factor = float(deltas[k2].get("distance_factor", 1.0))

            l_copy["distance_factor"] = factor
            enriched_links.append(l_copy)

        self.graph_data = {
            "nodes": raw_graph_data.get("nodes", []),
            "links": enriched_links,
            "hard_filter_threshold": raw_graph_data.get("hard_filter_threshold", HARD_FILTERING_THRESHOLD)
        }

    def _on_pairwise_edit(self, change):
        edit = change["new"]
        if edit and "chunk_1" in edit and "chunk_2" in edit:
            self.sidecar.save_pairwise_delta(
                chunk_id_1=edit["chunk_1"],
                chunk_id_2=edit["chunk_2"],
                distance_factor=edit["distance_factor"]
            )
EOF